# Communication API

First look at the ADEME API: we get the DPE data (existing and new buildings) and compare the two datasets.

## 1. Request

Useful functions for the rest of the notebook.

In [19]:
import pandas as pd
import numpy as np
import requests
import json

In [20]:
url_api_dpe_existants = "https://data.ademe.fr/data-fair/api/v1/datasets/dpe03existant/lines"
url_api_dpe_neufs = "https://data.ademe.fr/data-fair/api/v1/datasets/dpe02neuf/lines"

In [55]:


def set_params(page:int=None,size=10000,select:str=None,q:str=None,q_field:str=None,qs:str=None,params:dict=None)-> dict:
    """ function set params for the api request
    @param page: int, page number to request
    @param select: str, fields to select
    @param q: str, query string
    @param q_field: str, field to query
    @param qs: str, query string
    @param params: dict, additional parameters
    @return: dict, parameters for the api request
    """
    if params is None:
        params={
            "page": page if page is not None else None,
            "size": size if size is not None else None,
            "select": select if select is not None else None,
            "q": q if q is not None else None,
        "q_fields": q_field if q_field is not None else None,
        "qs": qs if qs is not None else None
    }
    return params

def req_api(url:str,size=10000,page:int=None,select:str=None,q:str=None,q_field:str=None,qs:str=None,params:dict=None)-> pd.DataFrame:
    """ function req api and return a dataframe with the data
    @param url: str, url of the api
    @param page: int, page number to request
    @param select: str, fields to select
    @param q: str, query string
    @param q_field: str, field to query
    @param qs: str, query string
    @param params: dict, additional parameters
    @return: pd.DataFrame, dataframe with the data
    """
    params = set_params(page=page,select=select,q=q,q_field=q_field,qs=qs,params=params)
        
    rep = requests.get(url, params=params)
    try:
        rep.raise_for_status()
    except requests.exceptions.HTTPError as e:
        print(f"Erreur lors de la requête : {rep.status_code} - {e}")
        return pd.DataFrame()
    if rep.status_code == 200:
        content = rep.json()
        df = pd.DataFrame(content['results'])
        print(f"nb line : {content['total']}")
        print(f"size data collected : {len(df)}, {len(df.columns) if not df.empty else 0}")
        return df

def get_nb_lines(url:str,page:int=None,size=10000,select:str=None,q:str=None,q_field:str=None,qs:str=None,params:dict=None)-> int:
    params = set_params(page=page,select=select,q=q,q_field=q_field,qs=qs,params=params)
    rep = requests.get(url, params=params)
    try:
        rep.raise_for_status()
    except requests.exceptions.HTTPError as e:
        print(f"Erreur lors de la requête : {rep.status_code} - {e}")
        return 0
    if rep.status_code == 200:
        content = rep.json()
        return content['total']

## 2. API test

### 2.1 Unique columns of each dataset


In [22]:
params = {
    "page": 1,
    "size": 10000,
}
df_dpe_existants = req_api(url_api_dpe_existants, params=params)
print(f"Nb unique columns for 'dpe existant' : {df_dpe_existants.columns.unique().value_counts().sum()}")

nb line : 15705770
size data collected : 10000, 218
Nb unique columns for 'dpe existant' : 218


In [23]:
df_dpe_neufs = req_api(url_api_dpe_neufs, params=params)
print(f"Nb unique columns for 'dpe neuf' : {df_dpe_neufs.columns.unique().value_counts().sum()}")

nb line : 1437707
size data collected : 10000, 191
Nb unique columns for 'dpe neuf' : 191


In [24]:
def compare_columns(df1: pd.DataFrame, df2: pd.DataFrame) -> None:
    """ function to compare columns of two dataframes
    @param df1: pd.DataFrame, first dataframe
    @param df2: pd.DataFrame, second dataframe
    """
    unique_columns = df1.columns.unique().tolist() + df2.columns.unique().tolist()
    print("nb unique columns in both datasets : ", len(set(unique_columns)))

    nb_columns_df1_not_in_df2 = len(set(df1.columns.unique().tolist()) - set(df2.columns.unique().tolist()))
    print("nb columns in df1 not in df2 : ", nb_columns_df1_not_in_df2)
   
    print("*" * 10)

    nb_columns_df2_not_in_df1 = len(set(df2.columns.unique().tolist()) - set(df1.columns.unique().tolist()))
    print("nb columns in df2 not in df1 : ", nb_columns_df2_not_in_df1)

def print_compare_columns_details(df1: pd.DataFrame, df2: pd.DataFrame) -> None:
    """ 
    @param df1: pd.DataFrame, first dataframe
    @param df2: pd.DataFrame, second dataframe
    @param file_path: str, path to the output file
    """
    columns_df1_not_in_df2 = set(df1.columns.unique().tolist()) - set(df2.columns.unique().tolist())
    columns_df2_not_in_df1 = set(df2.columns.unique().tolist()) - set(df1.columns.unique().tolist())

    print(f"Columns in df1 not in df2 : {len(columns_df1_not_in_df2)}")
    print(f"Columns name: ")
    for col in columns_df1_not_in_df2:
        print(f"- {col}")

    print("*" * 10)
    
    print(f"Columns in df2 not in df1 : {len(columns_df2_not_in_df1)}")
    print(f"Columns name: ")
    for col in columns_df2_not_in_df1:
        print(f"- {col}")


In [25]:
compare_columns(df_dpe_existants, df_dpe_neufs)


nb unique columns in both datasets :  222
nb columns in df1 not in df2 :  31
**********
nb columns in df2 not in df1 :  4


In [26]:
print_compare_columns_details(df_dpe_existants, df_dpe_neufs)

Columns in df1 not in df2 : 31
Columns name: 
- usage_generateur_n2_installation_n2
- date_installation_generateur_n1_ecs_n1
- qualite_isolation_plancher_haut_toit_terrasse
- numero_rpls_logement
- conso_ef_generateur_n1_ecs_n1
- surface_totale_capteurs_pv
- production_ecs_solaire_installation_n1
- periode_installation_generateur_froid
- description_generateur_chauffage_n2_installation_n2
- qualite_isolation_plancher_haut_comble_perdu
- description_generateur_chauffage_n1_installation_n1
- facteur_couverture_solaire_n1
- description_generateur_chauffage_n2_installation_n1
- type_generateur_n2_installation_n2
- conso_chauffage_generateur_n2_installation_n2
- apport_interne_saison_froide
- conso_chauffage_generateur_n2_installation_n1
- type_energie_climatisation
- nombre_module
- apport_solaire_saison_chauffe
- type_energie_generateur_n2_installation_n2
- description_generateur_chauffage_n1_installation_n2
- facteur_couverture_solaire_installation_chauffage_n1
- numero_immatriculation_c

**Result:** the two datasets do not have exactly the same columns.

Only in **DPE existants** (31 columns):
- `electricite_pv_autoconsommee`
- `description_generateur_chauffage_n1_installation_n2`
- `type_generateur_n2_installation_n1`
- `surface_totale_capteurs_pv`
- `type_energie_generateur_n2_installation_n1`
- `usage_generateur_n2_installation_n2`
- `date_installation_generateur_n1_ecs_n1`
- `type_ventilation`
- `systeme_production_electricite_origine_renouvelable`
- `usage_generateur_n2_installation_n1`
- `conso_ef_generateur_n1_ecs_n1`
- `numero_immatriculation_copropriete`
- `facteur_couverture_solaire_n1`
- `periode_installation_generateur_froid`
- `surface_ventilee`
- `type_energie_climatisation`
- `production_ecs_solaire_installation_n1`
- `qualite_isolation_plancher_haut_comble_perdu`
- `description_generateur_chauffage_n2_installation_n1`
- `numero_rpls_logement`
- `conso_chauffage_generateur_n1_installation_n1`
- `conso_chauffage_generateur_n2_installation_n1`
- `qualite_isolation_plancher_haut_toit_terrasse`
- `cop_generateur_n1_ecs_n1`
- `type_generateur_n2_installation_n2`
- `nombre_module`
- `description_generateur_chauffage_n2_installation_n2`
- `presence_production_pv`
- `conso_chauffage_generateur_n2_installation_n2`
- `type_energie_generateur_n2_installation_n2`
- `conso_chauffage_generateur_n1_installation_n2`
- `apport_solaire_saison_chauffe`
- `apport_interne_saison_chauffe`
- `apport_interne_saison_froide`
- `description_generateur_chauffage_n1_installation_n1`
- `apport_solaire_saison_froide`

Only in **DPE neufs** (4 columns):
- `deperditions_totales_batiment`
- `besoin_ecs_logement`
- `besoin_ecs_batiment`
- `deperditions_totales_logement`

### 2.2 Concat data

We keep only the columns common to both datasets.

In [27]:
# Find the common columns between the two datasets
common_columns = set(df_dpe_existants.columns.unique().tolist()).intersection(set(df_dpe_neufs.columns.unique().tolist()))
print(f"Common columns between the two datasets: {len(common_columns)}")
print(f"Columns name: ")
for col in common_columns:
    print(f"- {col}")

Common columns between the two datasets: 187
Columns name: 
- emission_ges_5_usages_energie_n2
- typologie_logement
- type_energie_n3
- configuration_installation_chauffage_n2
- description_generateur_froid
- type_batiment
- adresse_brut
- periode_construction
- nom_commune_brut
- nom_residence
- deperditions_renouvellement_air
- configuration_installation_chauffage_n1
- position_logement_dans_immeuble
- etiquette_dpe
- type_installation_chauffage_n1
- cout_ecs_energie_n2
- emission_ges_5_usages_par_m2
- complement_adresse_batiment
- deperditions_ponts_thermiques
- categorie_enr
- nombre_niveau_immeuble
- numero_dpe
- conso_auxiliaires_ep
- emission_ges_5_usages_energie_n1
- logement_traversant
- conso_ecs_ef_energie_n3
- _score
- type_installation_ecs_n1
- nombre_niveau_logement
- surface_habitable_desservie_par_installation_ecs_n1
- configuration_installation_ecs_n1
- description_generateur_n1_ecs_n1
- conso_chauffage_ef_energie_n1
- surface_ventilee
- deperditions_baies_vitrees
- da

In [28]:
df_dpe_existants['_id'].head(2)

0    8f6d7ec59cb91e4a79c294d5789c74bad817fd9d79ed29...
1    f27777e73bf74a3f5c31195229ee94ca6ddd2e4bc7c043...
Name: _id, dtype: str

In [29]:
df_dpe_existants['numero_dpe'].head(2)

0    2611E0031228S
1    2659E0039476U
Name: numero_dpe, dtype: str

Useful columns for the concat:
- `numero_dpe`
- `_id`

### 2.3 Data type

In [30]:
df_dpe_existants.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Columns: 218 entries, configuration_installation_chauffage_n2 to facteur_couverture_solaire_saisi_n1
dtypes: float64(115), int64(4), object(1), str(98)
memory usage: 16.6+ MB


In [31]:
df_dpe_neufs.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Columns: 191 entries, numero_voie_ban to type_ventilation
dtypes: float64(101), int64(5), object(1), str(84)
memory usage: 14.6+ MB


In [32]:
def get_columns_details(df: pd.DataFrame) -> pd.DataFrame:
    """ function to get columns details of a dataframe
    @param df: pd.DataFrame, dataframe to get columns details
    @return: pd.DataFrame, dataframe with columns details
    """
    columns_details = []
    for col in df.columns.unique().tolist():
        col_details = {
            "column_name": col,
            "data_type": df[col].dtype,
            "nb_unique_values": df[col].nunique(),
            "nb_missing_values": df[col].isnull().sum(),
            "missing_values_percentage": df[col].isnull().sum() / len(df) * 100,
        }
        columns_details.append(col_details)
    return pd.DataFrame(columns_details)

In [33]:
df_details_neufs= get_columns_details(df_dpe_existants).sort_values(by="nb_missing_values", ascending=False)
df_details_existant= get_columns_details(df_dpe_existants).sort_values(by="nb_missing_values", ascending=False)

In [34]:
df_details_existant.head()

,column_name,data_type,nb_unique_values,nb_missing_values,missing_values_percentage
143,_score,object,0,10000,100.00
217,facteur_couverture_solaire_saisi_n1,float64,4,9994,99.94
209,numero_rpls_logement,str,16,9984,99.84
216,facteur_couverture_solaire_installation_chauff...,float64,3,9983,99.83
210,surface_tertiaire_immeuble,float64,6,9967,99.67


In [35]:
df_details_neufs.head()

,column_name,data_type,nb_unique_values,nb_missing_values,missing_values_percentage
143,_score,object,0,10000,100.00
217,facteur_couverture_solaire_saisi_n1,float64,4,9994,99.94
209,numero_rpls_logement,str,16,9984,99.84
216,facteur_couverture_solaire_installation_chauff...,float64,3,9983,99.83
210,surface_tertiaire_immeuble,float64,6,9967,99.67


Needs to be considered:
- `_score` : type object and no value


In [36]:
print("Columns with at least 500 missing values:")
for name, df in [("existant", df_details_existant), ("neufs", df_details_neufs)]:
    col_sus = ', '.join(df.loc[df['nb_missing_values'] >= 500, 'column_name'])
    print(f"{name}: {col_sus}")
    print(f"nb : {len(df.loc[df['nb_missing_values'] >= 500])}")


Columns with at least 500 missing values:
existant: _score, facteur_couverture_solaire_saisi_n1, numero_rpls_logement, facteur_couverture_solaire_installation_chauffage_n1, surface_tertiaire_immeuble, usage_generateur_n2_installation_n2, description_generateur_chauffage_n2_installation_n2, conso_chauffage_generateur_n2_installation_n2, type_energie_generateur_n2_installation_n2, type_generateur_n2_installation_n2, numero_immatriculation_copropriete, electricite_pv_autoconsommee, production_ecs_solaire_installation_n1, facteur_couverture_solaire_n1, nombre_module, conso_ecs_ef_energie_n3, cout_total_5_usages_energie_n3, conso_chauffage_ef_energie_n3, emission_ges_5_usages_energie_n3, cout_ecs_energie_n3, emission_ges_chauffage_energie_n3, cout_chauffage_energie_n3, emission_ges_ecs_energie_n3, type_energie_n3, conso_5_usages_ef_energie_n3, surface_totale_capteurs_pv, typologie_logement, position_logement_dans_immeuble, cop_generateur_n1_ecs_n1, type_energie_climatisation, description_ge

At least 107 columns with at least 50% missing values. 
Those columns are the same for both df.

## 3. Departement exploration

**Goal**: Estimate departement data available

In [47]:
df_departements = pd.read_csv("../data/raw/codes_departements.csv",dtype="str")

In [48]:
df_departements["nb_lignes"] = np.nan

In [ ]:
for index, cp in df_departements["code_departement"].items():
    nb = get_nb_lines(url_api_dpe_existants,page=1,size=10000, qs=f"code_departement_ban:{cp}")
    print(f"code departement : {cp}, nb lignes : {nb}")
    df_departements.loc[index, "nb_lignes"] = nb



code departement : 01, nb lignes : 120523
code departement : 02, nb lignes : 98305
code departement : 03, nb lignes : 72398
code departement : 04, nb lignes : 39624
code departement : 05, nb lignes : 41362
code departement : 06, nb lignes : 299832
code departement : 07, nb lignes : 58112
code departement : 08, nb lignes : 60357
code departement : 09, nb lignes : 31491
code departement : 10, nb lignes : 65088
code departement : 11, nb lignes : 83029
code departement : 12, nb lignes : 49912
code departement : 13, nb lignes : 482802
code departement : 14, nb lignes : 182647
code departement : 15, nb lignes : 24478
code departement : 16, nb lignes : 69599
code departement : 17, nb lignes : 152384
code departement : 18, nb lignes : 68001
code departement : 19, nb lignes : 44186
code departement : 2A, nb lignes : 27089
code departement : 2B, nb lignes : 26206
code departement : 21, nb lignes : 132139
code departement : 22, nb lignes : 108130
code departement : 23, nb lignes : 22790
code depa

-> script in `reports/exploration.py`, data : `raw/codes_departements_nb_lignes.csv`

## 4. Conclusion

- **Data access:** the ADEME API works for both datasets (`dpe03existant`: ~15.6M rows, `dpe02neuf`: ~1.4M rows). The analysis was run on a sample of 1000 rows per dataset.
- **Structure:** 218 columns for existing buildings, 191 for new buildings. 187 columns are common to both. 31 columns exist only in *existants* (secondary generators, PV, ventilation, solar gains...) and 4 only in *neufs*.
There's a total of 222 columns.
- **Merging:** the two datasets can be concatenated on the common columns, with `numero_dpe` / `_id` as identifiers.
- **Types:** mostly numeric (`float64`) and text (`str`) columns. `_score` is empty and can be dropped.
- **Missing values:** 65 columns have at least 50% missing values in the sample. Need to confirm the missing-value rates on a larger sample.


